WARNING: This notebook is not in use, it exists as it was used to develop process_code.py. To implement edits to the processing steps, changes MUST be made to process_code.py.
This code is the alpha for retrieving, merging, and cleaning the generation timeseries. If any changes are made, they are made to this code and copied to all others.

In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# For plotting to verify
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

gen_details = pd.read_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv')
hw_tseries = pd.read_csv('/scratch/ng72/ms5578/time_series/gen_hw_status.csv')
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
# Select start and end dates (Jul 2009 - Jun 2024)
sdate, edate = '2009-07-01','2024-06-30'

In [4]:
# Select frequency of timeseries (hourly or daily)
mode = 'hourly'

In [5]:
# Select region or fuel type (optional)

def select_group(gen_details, state=None, ftype=None):
    if state is not None and ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[(gen_details['region'] == state) & (gen_details['fuel_source_primary'].isin(ftype))]
        else:
            groups = gen_details.groupby(['region', 'fuel_source_primary'])
            grp = groups.get_group((state, ftype))
    elif state is not None:
        groups = gen_details.groupby('region')
        grp = groups.get_group(state)
    elif ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[gen_details['fuel_source_primary'].isin(ftype)]
        else:
            groups = gen_details.groupby('fuel_source_primary')
            grp = groups.get_group(ftype)
    else:
        grp = gen_details

    return grp

info = select_group(gen_details,ftype=['Wind','Solar']).copy()

The functions below retrieve the timeseries for the specified dates.
The data is resampled to daily if specified above. The df is merged with the generation information.

In [6]:
def process_group(grp, gen_fpath, hw_tseries, start_date=None, end_date=None, mode='daily'):

    # Sanitize DUIDs and build file paths
    safe_duids = [duid.replace("/", "_").replace("\\", "_") for duid in grp['DUID']]
    gen_locs = [f"{gen_fpath}/{duid}.csv" for duid in safe_duids]
    dfs = [pd.read_csv(fp, dtype='object') for fp in gen_locs if os.path.exists(fp)]
    print(f"Loaded {len(dfs)} CSV file(s) out of {len(gen_locs)} expected.")

    if not dfs:
        raise ValueError("no files to load.")

    # Concatenate and clean header rows
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # Type conversions
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)

    # Filter by date
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time']).dt.normalize()
    hw_tseries = hw_tseries.set_index('time').sort_index()
    hw_tseries = hw_tseries.loc[start_date:end_date]

    if mode == 'daily':
        # Aggregate dfs to daily
        agg_func = {'TOTALMWh': 'sum', 'TOTALCLEARED': 'sum', 'AGCSTATUS': 'max'}
        dfs_daily = dfs.groupby(['DUID', pd.Grouper(freq='1D')]).agg(agg_func).reset_index()
        hw_tseries_daily = hw_tseries.reset_index()
        # Normalize time columns to midnight for exact matching
        dfs_daily['time'] = pd.to_datetime(dfs_daily['time']).dt.normalize()
        hw_tseries_daily['time'] = pd.to_datetime(hw_tseries_daily['time']).dt.normalize()
        # Merge on DUID and time
        merged = pd.merge(
            dfs_daily,
            hw_tseries_daily,
            on=['DUID', 'time'],
            how='left'
        )
    elif mode == 'hourly':
        dfs_hourly = dfs.reset_index()
        # Create full hourly time index for each DUID
        duids = dfs_hourly['DUID'].unique()
        hourly_range = pd.date_range(start=start_date, end=end_date, freq='1h')
        full_index = pd.MultiIndex.from_product([duids, hourly_range], names=['DUID', 'time'])
        hw_hourly = pd.DataFrame(index=full_index).reset_index()
        hw_hourly['date'] = hw_hourly['time'].dt.normalize()
        # Prepare daily hw_tseries for merging
        hw_tseries_daily = hw_tseries.reset_index()
        hw_tseries_daily['date'] = pd.to_datetime(hw_tseries_daily['time']).dt.normalize()
        # Merge daily values onto hourly DataFrame by DUID and date
        hw_tseries_broadcast = pd.merge(
            hw_hourly,
            hw_tseries_daily.drop(columns='time'),
            on=['DUID', 'date'],
            how='left'
        ).drop(columns='date')
        # Merge on DUID and time
        merged = pd.merge(
            dfs_hourly,
            hw_tseries_broadcast,
            on=['DUID', 'time'],
            how='left'
        )
    else:
        raise ValueError("mode must be 'daily' or 'hourly'")

    merged = merged.dropna(how='all')

    # Jitter coordinates for plotting
    def jitter(group):
        duplicates = group.groupby(['lat', 'lon']).cumcount()
        np.random.seed(42)
        jitter_strength = 0.05
        group['lat_jittered'] = group['lat'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates
        group['lon_jittered'] = group['lon'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates
        return group

    grp = jitter(grp)

    df = merged.merge(
        grp[['DUID', 'fuel_source_primary', 'region', 'lat_jittered', 'lon_jittered']],
        on='DUID',
        how='left'
    )

    return df.reset_index(drop=True)

In [7]:
# Make sure mode is set correctly to daily or hourly
df = process_group(info, gen_fpath, hw_tseries, sdate, edate, mode=mode)
df

Loaded 164 CSV file(s) out of 216 expected.


,time,DUID,TOTALMWh,TOTALCLEARED,AGCSTATUS,lat,lon,tas,EHF_val,HW_EHF_avg,HW_EHF_peak,EHF_flag,tas_3d_avg,tas_3d_peak,event_group,HW_event_day,fuel_source_primary,region,lat_jittered,lon_jittered
0,2009-07-01,HALLWF2,47.355833,45.185000,0,-33.33,138.75,281.996826,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind,SA1,-33.334896,138.765402
1,2009-07-01,HALLWF1,84.668333,84.755833,0,-33.33,138.75,281.996826,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind,SA1,-33.348652,138.751659
2,2009-07-01,LKBONNY2,25.312352,27.395000,0,-37.84,140.40,285.555908,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind,SA1,-37.814894,140.419499
3,2009-07-01,CLEMGPWF,15.061319,14.305833,0,-33.55,138.09,285.563721,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind,SA1,-33.508568,138.119175
4,2009-07-01,SNOWTWN1,79.533435,79.000000,0,-33.66,138.20,284.832031,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind,SA1,-33.711237,138.145557
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7262996,2024-06-30,WDGPH1,0.000000,0.000000,0,-26.95,150.63,291.742920,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Solar,QLD1,-26.958390,150.678910
7262997,2024-06-30,GUNNING1,1.965914,1.995083,0,-34.65,149.42,276.201172,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind,NSW1,-34.697734,149.386126
7262998,2024-06-30,HALLWF1,4.745833,4.810833,0,-33.33,138.75,279.057373,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Wind,SA1,-33.348652,138.751659
7262999,2024-06-30,GNNDHSF1,0.000000,0.000000,0,-30.91,150.30,285.236328,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Solar,NSW1,-30.949586,150.345351


The following section contains all the data cleaning functions and their execution. These should be selected based on technology type and research question.

These can be used for almost all code

In [8]:
def sel_months(df, months=[12,1,2]):
    df = df[df['time'].dt.month.isin(months)]
    return df

In [9]:
def min_heatwave_days(df, min_days=20):
    """
    Filters the DataFrame to include only DUIDs with at least `min_days`
    of unique heatwave days (EHF_flag == 1).
    
    Works with hourly or higher-frequency data by counting unique *days*.

    Parameters:
        df (pd.DataFrame): Input DataFrame with columns ['DUID', 'time', 'EHF_flag']
        min_days (int): Minimum number of unique heatwave days required

    Returns:
        pd.DataFrame: Filtered DataFrame
    """
    df = df.copy()
    df['time'] = pd.to_datetime(df['time'])
    
    # Normalize to midnight → one unique timestamp per day
    df['date'] = df['time'].dt.normalize()

    # Count unique heatwave days per DUID
    heatwave_days = (
        df[df['EHF_flag'] == 1]
        .groupby('DUID')['date']
        .nunique()
    )

    # Keep only DUIDs meeting the threshold
    valid_duids = heatwave_days[heatwave_days >= min_days].index
    
    return df[df['DUID'].isin(valid_duids)].copy()


In [10]:
def remove_negatives(df):
    # Removes negatives from df, generally fine to use for all except hydro (which is outside scope).
    df = df[~((df['TOTALMWh'] < 0))]
    return df

These can be used for hourly data

In [11]:
def remove_solar_night(df):
    # For solar only: remove rows between 20:00 and 06:00
    solar_mask = df['fuel_source_primary'] == 'Solar'
    times = df.loc[solar_mask, 'time'].dt.time
    
    time_filter = ~(
        (times >= datetime.time(21, 0)) |  # 21:00 onwards
        (times < datetime.time(5, 0))      # before 05:00
    )
    
    # Keep all non-solar rows, and solar rows passing the time filter
    df = pd.concat([
        df[~solar_mask],
        df.loc[solar_mask].loc[time_filter]
    ])

    return df

In [12]:
def remove_wind_zeros(df, duid_col='DUID', value_col='TOTALMWh',
                      tech_col='fuel_source_primary', wind_label='Wind', threshold=None):
    # Mann-Whitney U tests are sensitive to excessive zeros, so any DUIDs with over 40% 0s are excluded from the data.
    """
    Remove *all rows* for wind DUIDs where the percentage of zeros 
    exceeds the given threshold. Non-wind DUIDs are left untouched.
    
    Parameters:
        df (pd.DataFrame): Input dataframe
        duid_col (str): Column name for grouping (default 'DUID')
        value_col (str): Column name with numeric values (default 'TOTALMWh')
        tech_col (str): Column that identifies technology type (default 'fuel_source_primary')
        wind_label (str): Label used for wind in tech_col (default 'Wind')
        threshold (float): Maximum allowed percentage of zeros (default 5)
    
    Returns:
        pd.DataFrame: Filtered dataframe
    """
    
    # work only on wind rows
    wind_df = df[df[tech_col] == wind_label]
    
    # calculate % of zeros per wind DUID
    percent_zeros = (
        wind_df.groupby(duid_col)[value_col]
               .apply(lambda x: (x == 0).sum() / len(x) * 100)
    )
    
    # keep DUIDs below threshold
    keep_duids = percent_zeros[percent_zeros <= threshold].index
    
    # filter wind df
    wind_filtered = wind_df[wind_df[duid_col].isin(keep_duids)]
    
    # keep all non-wind rows
    non_wind_df = df[df[tech_col] != wind_label]
    
    # combine and return
    return pd.concat([non_wind_df, wind_filtered], ignore_index=True)

These can be used for specific tech types

In [13]:
def clear_agc(df):
    # Removes days where AGCSTATUS is 0 and the plant is non-operational.
    # Generally ill-advised
    df = df[~((df['TOTALMWh'] < 0))].copy()
    mask = (df['fuel_source_primary'].isin([
            'Water', 'Natural Gas Pipeline', 'Black Coal', 'Coal Seam Methane',
            'Brown Coal', 'Diesel', 'Kerosene'
        ]) & (df['TOTALCLEARED'] <= 0))
    df.loc[mask, 'TOTALMWh'] = np.nan
    return df

In [14]:
df = sel_months(df)
df = remove_negatives(df)
df = remove_wind_zeros(df, threshold=40)
df = min_heatwave_days(df,20)

In [15]:
df

,time,DUID,TOTALMWh,TOTALCLEARED,AGCSTATUS,lat,lon,tas,EHF_val,HW_EHF_avg,...,EHF_flag,tas_3d_avg,tas_3d_peak,event_group,HW_event_day,fuel_source_primary,region,lat_jittered,lon_jittered,date
0,2014-12-16 00:00:00,NYNGAN1,0.000000,0.000000,0,-31.57,147.11,303.112061,0.000000,NaN,...,0.0,NaN,NaN,NaN,NaN,Solar,NSW1,-31.552740,147.085352,2014-12-16
1,2014-12-16 01:00:00,NYNGAN1,0.000000,0.000000,0,-31.57,147.11,303.112061,0.000000,NaN,...,0.0,NaN,NaN,NaN,NaN,Solar,NSW1,-31.552740,147.085352,2014-12-16
2,2014-12-16 02:00:00,NYNGAN1,0.000000,0.000000,0,-31.57,147.11,303.112061,0.000000,NaN,...,0.0,NaN,NaN,NaN,NaN,Solar,NSW1,-31.552740,147.085352,2014-12-16
3,2014-12-16 03:00:00,NYNGAN1,0.000000,0.000000,0,-31.57,147.11,303.112061,0.000000,NaN,...,0.0,NaN,NaN,NaN,NaN,Solar,NSW1,-31.552740,147.085352,2014-12-16
4,2014-12-16 04:00:00,NYNGAN1,0.000000,0.000000,0,-31.57,147.11,303.112061,0.000000,NaN,...,0.0,NaN,NaN,NaN,NaN,Solar,NSW1,-31.552740,147.085352,2014-12-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1838012,2024-02-29 23:00:00,NBHWF1,58.366667,58.308333,0,-33.22,138.75,298.397705,6.395517,3.621937,...,1.0,299.352485,300.239095,76.0,3.0,Wind,SA1,-33.252576,138.712289,2024-02-29
1838013,2024-02-29 23:00:00,BULGANA1,76.704167,69.694381,0,-37.07,142.93,293.941406,0.000000,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,VIC1,-37.118103,142.960530,2024-02-29
1838015,2024-02-29 23:00:00,WOODLWN1,1.732937,1.992333,0,-35.09,149.53,295.635986,1.000000,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,NSW1,-35.086060,149.571606,2024-02-29
1838016,2024-02-29 23:00:00,MACARTH1,0.000000,0.000000,0,-38.06,142.16,290.064453,0.000000,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,VIC1,-38.064804,142.184360,2024-02-29


In [1]:
import os
import datetime
import time
import pandas as pd
import numpy as np
import dask.dataframe as dd
import psutil
import logging

# ----------------------
# Constants for filepaths
# ----------------------
GEN_INFO_FP = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv"
HW_FP = "/scratch/ng72/ms5578/time_series/gen_hw_status.csv"
GEN_FPATH = "/scratch/ng72/ms5578/time_series/nem_generation"

# ----------------------
# Dask client singleton (auto-starts on import)
# ----------------------
_dask_client = None

def get_dask_client(
    workload_type="io",
    max_workers=None,
    reserve_mem_gb=50,
    max_mem_gb=None,
    dashboard=True,
    dashboard_address=":8787"
):
    global _dask_client
    if _dask_client is not None:
        print(f"Dask dashboard: {_dask_client.dashboard_link}")
        return _dask_client

    from dask.distributed import Client
    logical_cores = psutil.cpu_count(logical=True)
    total_memory_gb = psutil.virtual_memory().total / 1e9
    if max_workers is None:
        max_workers = logical_cores
    if max_mem_gb is None:
        max_mem_gb = total_memory_gb
    usable_mem_gb = max_mem_gb - reserve_mem_gb
    if workload_type == "cpu":
        threads_per_worker = 1
        n_workers = min(max_workers, logical_cores)
    elif workload_type == "io":
        threads_per_worker = 8
        n_workers = max(1, logical_cores // threads_per_worker)
    else:  # "mixed"
        threads_per_worker = 4
        n_workers = max(1, logical_cores // threads_per_worker)
    memory_per_worker = usable_mem_gb // n_workers
    logging.getLogger("distributed.worker.memory").setLevel(logging.ERROR)
    logging.getLogger("dask").setLevel(logging.ERROR)
    _dask_client = Client(
        n_workers=n_workers,
        threads_per_worker=threads_per_worker,
        memory_limit=f"{int(memory_per_worker)}GB",
        dashboard_address=dashboard_address if dashboard else None
    )
    print(f"Dask dashboard: {_dask_client.dashboard_link}")
    return _dask_client

get_dask_client(dashboard=True, dashboard_address=":8787")

# ----------------------
# Selection helpers
# ----------------------
def select_group(gen_details_dd, state=None, ftype=None):
    """Selects a group from the Dask DataFrame."""
    grp_dd = gen_details_dd
    if state is not None:
        grp_dd = grp_dd[grp_dd['region'] == state]
    
    if ftype is not None:
        if isinstance(ftype, list):
            grp_dd = grp_dd[grp_dd['fuel_source_primary'].isin(ftype)]
        else:
            grp_dd = grp_dd[grp_dd['fuel_source_primary'] == ftype]
            
    return grp_dd

# ----------------------
# Processing
# ----------------------
# ----------------------
# Processing
# ----------------------
# ----------------------
# Processing
# ----------------------
def process_group_dask(grp_pd, grp_dd, gen_fpath, hw_tseries_dd, start_date, end_date, mode="daily"):
    timings = {}
    print("--- Starting Dask-Native Process ---")
    
    # 1. Get safe DUIDs from the computed pandas DataFrame
    t0 = time.time()
    safe_duids = set(duid.replace("/", "_").replace("\\", "_") for duid in grp_pd['DUID'])
    timings['duid_setup'] = time.time() - t0

    # 2. Bulk Dask read_csv
    t0 = time.time()
    dfs = dd.read_csv(
        f"{gen_fpath}/*.csv",
        dtype={'DUID': 'string'},
        assume_missing=True,
        blocksize="256MB"
    )
    timings['csv_bulk_read'] = time.time() - t0

    # 3. Filter DUIDs
    t0 = time.time()
    dfs = dfs[dfs['DUID'].isin(safe_duids)]
    dfs = dfs[dfs['DUID'] != 'DUID']
    timings['filter_and_clean'] = time.time() - t0

    # 4. Type conversions and dropping nulls
    t0 = time.time()
    dfs['time'] = dd.to_datetime(dfs['time'], errors='coerce')
    dfs['TOTALCLEARED'] = dd.to_numeric(dfs['TOTALCLEARED'], errors='coerce')
    dfs['TOTALMWh'] = dd.to_numeric(dfs['TOTALMWh'], errors='coerce')
    dfs['AGCSTATUS'] = dd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    dfs = dfs.dropna(subset=['time', 'DUID'])
    timings['type_conversion_and_dropna'] = time.time() - t0
    
    # 5. Filter by date
    t0 = time.time()
    dfs = dfs[(dfs['time'] >= start_date) & (dfs['time'] <= end_date)]
    timings['date_filter'] = time.time() - t0

    # 6. Prepare heatwave time series
    t0 = time.time()
    hw_tseries_dd['time'] = dd.to_datetime(hw_tseries_dd['time']).dt.normalize()
    hw_tseries_dd = hw_tseries_dd.set_index('time').loc[start_date:end_date]
    timings['hw_tseries_filter'] = time.time() - t0

    # 7. Aggregation/groupby
    t0 = time.time()
    if mode == 'hourly':
        print("Setting index and sorting data by time... (This may take a while)")
        # --- FIX: Use a new variable for the indexed dataframe ---
        dfs_indexed = dfs.set_index('time')
        dfs_hourly = dfs_indexed.reset_index()
        # ---------------------------------------------------------
        hw_daily_dd = hw_tseries_dd.reset_index()
        hw_daily_dd['date'] = hw_daily_dd['time'].dt.normalize()
        dfs_hourly['date'] = dfs_hourly['time'].dt.normalize()
        merged = dd.merge(
            dfs_hourly, hw_daily_dd.drop(columns='time'), on=['DUID', 'date'], how='left'
        ).drop(columns='date')
        timings['aggregate_hourly'] = time.time() - t0
    else: # daily mode
        agg_func = {'TOTALMWh': 'sum', 'TOTALCLEARED': 'sum', 'AGCSTATUS': 'max'}
        dfs['date'] = dfs['time'].dt.normalize()
        dfs_daily = dfs.groupby(['DUID', 'date']).agg(agg_func).reset_index()
        hw_daily_dd = hw_tseries_dd.reset_index()
        hw_daily_dd['time'] = hw_daily_dd['time'].dt.normalize()
        
        merged = dd.merge(dfs_daily, hw_daily_dd, left_on=['DUID', 'date'], right_on=['DUID', 'time'], how='left')
        merged = merged.drop(columns='time') 
        merged = merged.rename(columns={'date': 'time'})
        
        timings['aggregate_daily'] = time.time() - t0

    # 8. Drop all-NA rows
    merged = merged.dropna(how='all')
    
    # 9. Jitter for plotting
    t0 = time.time()
    grp_pd_jittered = grp_pd.groupby(['lat', 'lon'], group_keys=False).apply(
        lambda g: g.assign(
            lat_jittered=g['lat'] + np.random.uniform(-0.05, 0.05, size=len(g)) * np.arange(len(g)),
            lon_jittered=g['lon'] + np.random.uniform(-0.05, 0.05, size=len(g)) * np.arange(len(g))
        )
    )
    timings['jitter'] = time.time() - t0

    # 10. Final merge with generator details
    t0 = time.time()
    grp_pd_jittered_dd = dd.from_pandas(grp_pd_jittered, npartitions=1)
    merged = merged.merge(
        grp_pd_jittered_dd[['DUID', 'fuel_source_primary', 'region', 'lat_jittered', 'lon_jittered']],
        on='DUID',
        how='left'
    )
    timings['final_merge'] = time.time() - t0

    # 11. Compute the final result
    print("Starting final Dask compute...")
    t0 = time.time()
    df_computed = merged.compute()
    timings['compute'] = time.time() - t0
    print("Dask compute finished.")
    
    print("\n--- DASK TIMING REPORT ---")
    for k, v in timings.items():
        print(f"{k}: {v:.2f} seconds")
    print("--- END REPORT ---\n")
    
    return df_computed.reset_index(drop=True), grp_pd_jittered.reset_index(drop=True)

# ----------------------
# Filters (pandas)
# ----------------------
def sel_months(df, months=[12,1,2]):
    return df[df['time'].dt.month.isin(months)]

def min_heatwave_days(df, min_days=20):
    df = df.copy()
    df['date'] = pd.to_datetime(df['time']).dt.normalize()
    heatwave_days = (
        df[df['EHF_flag'] == 1]
        .groupby('DUID')['date']
        .nunique()
    )
    valid_duids = heatwave_days[heatwave_days >= min_days].index
    return df[df['DUID'].isin(valid_duids)].copy()

def remove_negatives(df):
    return df[df['TOTALMWh'] >= 0]

def remove_solar_night(df):
    solar_mask = df['fuel_source_primary'] == 'Solar'
    times = df.loc[solar_mask, 'time'].dt.time
    time_filter = ~(
        (times >= datetime.time(21, 0)) |
        (times < datetime.time(5, 0))
    )

    non_solar = df.loc[~solar_mask].copy()
    solar = df.loc[solar_mask].copy()
    solar = solar.loc[time_filter]

    return pd.concat([non_solar, solar], ignore_index=True)

def remove_wind_zeros(df, threshold=40):
    wind_df = df[df['fuel_source_primary'] == 'Wind']
    percent_zeros = (
        wind_df.groupby('DUID')['TOTALMWh']
               .apply(lambda x: (x == 0).sum() / len(x) * 100)
    )
    keep_duids = percent_zeros[percent_zeros <= threshold].index
    wind_filtered = wind_df[wind_df['DUID'].isin(keep_duids)]
    non_wind_df = df[df['fuel_source_primary'] != 'Wind']
    return pd.concat([non_wind_df, wind_filtered], ignore_index=True)

def clear_agc(df):
    # Removes days where AGCSTATUS is 0 and the plant is non-operational.
    # Generally ill-advised
    df = df[~((df['TOTALMWh'] < 0))].copy()
    mask = (df['fuel_source_primary'].isin([
            'Water', 'Natural Gas Pipeline', 'Black Coal', 'Coal Seam Methane',
            'Brown Coal', 'Diesel', 'Kerosene'
        ]) & (df['TOTALCLEARED'] <= 0))
    df.loc[mask, 'TOTALMWh'] = np.nan
    return df

# ----------------------
# Main entry point
# ----------------------
def load_generation_data(
    sdate,
    edate,
    mode="daily",
    state=None,
    ftype=None,
    apply_sel_months=False,
    months=[12,1,2],
    apply_remove_negatives=False,
    apply_remove_wind_zeros=False,
    wind_zero_threshold=40,
    apply_solar_night=False,
    apply_min_heatwave_days=False,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
):
    t0 = time.time()
    
    gen_details_dtype = {
        'DUID': 'string',
        'max_cap_mw': 'object',
        'reg_cap_mw': 'object',
        'reg_cap_mw_nmap': 'object'
    }
    gen_details_dd = dd.read_csv(GEN_INFO_FP, dtype=gen_details_dtype)
    
    hw_tseries_dd = dd.read_csv(HW_FP, dtype={'DUID': 'string'})
    
    print(f"Read gen_details & hw_tseries with Dask: {time.time()-t0:.2f} sec")

    t1 = time.time()
    grp_dd = select_group(gen_details_dd, state=state, ftype=ftype)
    print(f"Select group: {time.time()-t1:.2f} sec")

    t2 = time.time()
    grp_pd = grp_dd.compute() 
    df, grp = process_group_dask(grp_pd, grp_dd, GEN_FPATH, hw_tseries_dd, sdate, edate, mode)
    print(f"Process group: {time.time()-t2:.2f} sec")

    t3 = time.time()
    if apply_sel_months:
        df = sel_months(df, months=months)
    if apply_remove_negatives:
        df = remove_negatives(df)
    if apply_remove_wind_zeros:
        df = remove_wind_zeros(df, threshold=wind_zero_threshold)
    if apply_solar_night:
        df = remove_solar_night(df)
    if apply_min_heatwave_days:
        df = min_heatwave_days(df, min_days=min_heatwave_days_threshold)
    if apply_clear_agc:
        df = clear_agc(df)
    print(f"Filters: {time.time()-t3:.2f} sec")

    # --- NEW FIX: Synchronize the 'grp' DataFrame with the final 'df' ---
    # This ensures the generator info exactly matches the final, filtered data.
    if not df.empty:
        final_duids = df['DUID'].unique()
        grp = grp[grp['DUID'].isin(final_duids)].copy()
    else:
        # If df is empty, grp should also be empty.
        grp = grp.iloc[0:0].copy()
    # --------------------------------------------------------------------

    return df, grp

Dask dashboard: /proxy/8787/status


2025-09-25 09:27:31,169 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a03d03119efde2921c2fae0cc82e2976 initialized by task ('shuffle-transfer-a03d03119efde2921c2fae0cc82e2976', 230) executed on worker tcp://127.0.0.1:33355
2025-09-25 09:27:35,805 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle cec16e5987ea175d405a94b7fa91af19 initialized by task ('shuffle-transfer-cec16e5987ea175d405a94b7fa91af19', 0) executed on worker tcp://127.0.0.1:41313
2025-09-25 09:27:41,754 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle cec16e5987ea175d405a94b7fa91af19 deactivated due to stimulus 'task-finished-1758756461.742162'
2025-09-25 09:28:02,972 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a03d03119efde2921c2fae0cc82e2976 deactivated due to stimulus 'task-finished-1758756482.902122'
2025-09-25 09:28:53,107 - distributed.nanny - WARNING - Restarting worker
2025-09-25 09:28:53,132 - distributed.nanny - WARNING - Restarting worker
2025-09-25 09:28:53,13

In [2]:
df, info = load_generation_data(
    sdate="2020-07-01",
    edate="2024-06-30",
    mode="hourly",
    ftype=["Wind"],
    apply_remove_negatives=True,
    apply_remove_wind_zeros=True,
    wind_zero_threshold=40,
    apply_min_heatwave_days=True,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
)

Read gen_details & hw_tseries with Dask: 0.04 sec
Select group: 0.01 sec
--- Starting Dask-Native Process ---
Setting index and sorting data by time... (This may take a while)


/jobfs/150765768.gadi-pbs/ipykernel_2774322/2650690844.py:165: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grp_pd_jittered = grp_pd.groupby(['lat', 'lon'], group_keys=False).apply(


Starting final Dask compute...
Dask compute finished.

--- DASK TIMING REPORT ---
duid_setup: 0.00 seconds
csv_bulk_read: 0.35 seconds
filter_and_clean: 0.01 seconds
type_conversion_and_dropna: 0.03 seconds
date_filter_and_sort: 0.01 seconds
hw_tseries_filter: 5.21 seconds
aggregate_hourly: 0.02 seconds
jitter: 0.20 seconds
final_merge: 0.01 seconds
compute: 68.79 seconds
--- END REPORT ---

Process group: 78.52 sec
Filters: 2.02 sec


In [3]:
df

,time,DUID,TOTALMWh,TOTALCLEARED,AGCSTATUS,lat,lon,tas,EHF_val,HW_EHF_avg,...,EHF_flag,tas_3d_avg,tas_3d_peak,event_group,HW_event_day,fuel_source_primary,region,lat_jittered,lon_jittered,date
0,2020-07-01,ARWF1,40.966667,41.658333,0.0,-37.29,143.04,282.841797,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,VIC1,-37.238518,143.079431,2020-07-01
1,2020-07-01,BULGANA1,19.290833,18.644750,0.0,-37.07,142.93,283.568359,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,VIC1,-37.118103,142.960530,2020-07-01
5,2020-07-01,MACARTH1,172.821008,172.821167,0.0,-38.06,142.16,284.201172,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,VIC1,-38.064804,142.184360,2020-07-01
6,2020-07-01,HALLWF1,22.940000,22.956667,0.0,-33.33,138.75,284.282471,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,SA1,-33.348652,138.751659,2020-07-01
7,2020-07-01,BODWF1,36.038147,36.171833,0.0,-32.45,149.09,284.395752,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,NSW1,-32.414600,149.099795,2020-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2200934,2024-06-30,MACARTH1,0.000000,0.000000,0.0,-38.06,142.16,280.892578,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,VIC1,-38.064804,142.184360,2024-06-30
2200935,2024-06-30,DULAWF1,14.416667,14.400000,0.0,-26.62,149.86,291.332031,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,QLD1,-26.637084,149.871046,2024-06-30
2200939,2024-06-30,CTHLWF1,6.246282,6.180405,0.0,-42.13,146.67,272.984375,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,TAS1,-42.159931,146.708927,2024-06-30
2200940,2024-06-30,SAPHWF1,11.970322,12.045833,0.0,-29.70,151.40,285.264404,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,Wind,NSW1,-29.699726,151.411940,2024-06-30
